In [ ]:
import torch
import numpy as np
import pandas as pd
import ray
import transformers
import jupyter
from torch.utils.data import Dataset, DataLoader
from transformers import AutoConfig
import json
import wandb
import random
from tqdm import tqdm

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
qwen3 = AutoModelForCausalLM.from_pretrained(
	"Qwen/Qwen3-0.6B",
	device_map="cpu"
)
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, #Rank of low rank adaptation
    lora_alpha=16, #Scaling parameter (usually 2x rank)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], #Which modules to apply LoRA to (attention and MoE layers)
    lora_dropout=0.1,
    bias="none",
    inference_mode=False
)

training_args = TrainingArguments(
    output_dir="./mysterydungeonGPT/qwen3-lora-finetuned",
    num_train_epochs=1,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_steps=0.3,
    eval_strategy="steps",
    gradient_checkpointing=True,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    report_to="wandb"
)


training_config = {
    "model_name": "Qwen/Qwen3-0.6B",
    "lora_config":lora_config,
    "training_args":training_args,
    "data_collator":data_collator,
    "max_length": 1024,
    "train_split": 0.8,
    "eval_split": 0.2,
    "preprocessing": {
        "remove_empty_lines": True,
        "min_length": 4096,
        "max_length": 6144
    }
}

In [ ]:
from datasets import load_dataset

dataset = load_dataset("teamgas/mysterydungeondata")

In [ ]:
messages = [
    {"role": "user", "content": "Generate a medium difficulty..."},
    {"role": "assistant", "content:": "{'tiles': [...], ...}"}
]

In [ ]:
# Debug: Check what format map_array is in
example = dataset['train'][0]
print("Type:", type(example['map_array']))
print("First 200 chars:", str(example['map_array'])[:200])
print("Has ...:", '...' in str(example['map_array']))

In [ ]:
def format_map_for_training(map):
    room_count = map['room_count']
    complexity = map['complexity']
    difficulty = map['difficulty']

    enemies = json.loads(map['enemies'])
    gen_params = json.loads(map['generation_params'])

    # Try to get map_array, fallback to image extraction
    map_array = map.get('map_array')

    map_width = map.get('width', 56) 
    map_height = map.get('height', 32)
    
    # If map_array is a string with numpy format, extract from image instead
    if isinstance(map_array, str) and '...' in map_array:
        # Extract from image
        from PIL import Image
        import io
        
        image = map['image']
        if isinstance(image, Image.Image):
            # Convert image to numpy array
            img_array = np.array(image)
            img_height, img_width = img_array.shape[:2]
            
            # Extract map: black=wall (0), brown=floor (1), green=player (2), red=stairs (3)
            # Initialize with walls (0)
            map_array = np.zeros((img_height, img_width), dtype=np.uint8)
            
            # Brown floors: [139, 69, 19]
            floor_mask = (img_array[:, :, 0] == 139) & (img_array[:, :, 1] == 69) & (img_array[:, :, 2] == 19)
            map_array[floor_mask] = 1
            
            # Green player spawn: [0, 255, 0]
            player_mask = (img_array[:, :, 0] == 0) & (img_array[:, :, 1] == 255) & (img_array[:, :, 2] == 0)
            map_array[player_mask] = 2
            
            # Red stairs spawn: [255, 0, 0]
            stairs_mask = (img_array[:, :, 0] == 255) & (img_array[:, :, 1] == 0) & (img_array[:, :, 2] == 0)
            map_array[stairs_mask] = 3

            # Downsample from image size to map size
            # Calculate scaling factors
            scale_y = img_height / map_height
            scale_x = img_width / map_width
            
            # Downsample by taking the mode (most common value) in each tile region
            # Simple approach: sample at regular intervals
            downsampled = np.zeros((map_height, map_width), dtype=np.uint8)
            for y in range(map_height):
                for x in range(map_width):
                    # Get the corresponding region in the image
                    img_y_start = int(y * scale_y)
                    img_y_end = int((y + 1) * scale_y)
                    img_x_start = int(x * scale_x)
                    img_x_end = int((x + 1) * scale_x)
                    
                    # Get the region
                    region = map_array[img_y_start:img_y_end, img_x_start:img_x_end]
                    # Use the most common value (mode) in the region
                    if region.size > 0:
                        values, counts = np.unique(region, return_counts=True)
                        downsampled[y, x] = values[np.argmax(counts)]
            
            map_array = downsampled
        else:
            raise ValueError("Could not extract map from image")
    elif isinstance(map_array, str):
        # Try to parse as regular string (shouldn't happen but just in case)
        try:
            import ast
            parsed = ast.literal_eval(map_array)
            map_array = np.array(parsed)
        except:
            raise ValueError("Could not parse map_array string")

    player_x = map['player_spawn_x']
    player_y = map['player_spawn_y']
    stairs_x = map['stairs_spawn_x']
    stairs_y = map['stairs_spawn_y']

    templates = [
        # Direct/Simple
        "Generate a {difficulty} difficulty dungeon with {room_count} rooms",
        "Create a {difficulty} dungeon containing {room_count} rooms",
        "Design a {difficulty} level dungeon with {room_count} rooms",
        
        # With Complexity
        "Generate a {difficulty} difficulty dungeon with {room_count} rooms and complexity {complexity}",
        "Create a dungeon map: {difficulty} difficulty, {room_count} rooms, complexity {complexity}",
        "Design a {difficulty} dungeon with {room_count} rooms at complexity level {complexity}",
        
        # Structured/Technical
        "Generate dungeon: difficulty={difficulty}, room_count={room_count}, complexity={complexity}",
        "Create dungeon map with parameters: difficulty={difficulty}, rooms={room_count}, complexity={complexity}",
        "Dungeon specifications: {difficulty} difficulty, {room_count} rooms, complexity {complexity}",
        
        # Conversational
        "I need a {difficulty} dungeon with {room_count} rooms",
        "Please generate a {difficulty} difficulty dungeon containing {room_count} rooms",
        "Can you create a {difficulty} dungeon map with {room_count} rooms?",
        
        # Descriptive
        "Generate a {difficulty} difficulty dungeon featuring {room_count} rooms and complexity {complexity}",
        "Create a {difficulty} level dungeon map with {room_count} rooms, complexity set to {complexity}",
        "Design a {difficulty} dungeon containing {room_count} rooms with a complexity of {complexity}",
        
        # Game Context
        "For a roguelike game, generate a {difficulty} dungeon with {room_count} rooms",
        "Create a playable {difficulty} dungeon map with {room_count} rooms and complexity {complexity}",
        "Generate a {difficulty} dungeon layout: {room_count} rooms, complexity {complexity}",
        
        # Varied Phrasing
        "Build a {difficulty} dungeon consisting of {room_count} rooms",
        "Produce a {difficulty} difficulty dungeon map with {room_count} rooms",
        "Make a {difficulty} dungeon with {room_count} rooms, complexity {complexity}",
        
        # More Natural
        "A {difficulty} dungeon with {room_count} rooms, please",
        "Generate me a {difficulty} difficulty dungeon that has {room_count} rooms",
        "I want a {difficulty} dungeon map with {room_count} rooms and complexity {complexity}",
        
        # Compact
        "{difficulty} dungeon, {room_count} rooms, complexity {complexity}",
        "Dungeon: {difficulty}, {room_count} rooms, complexity {complexity}",
        "Map: {difficulty} difficulty, {room_count} rooms, {complexity} complexity",
    ]

    prompt = random.choice(templates).format(
        difficulty = difficulty,
        room_count = room_count,
        complexity = round(complexity, 2)
    )

    # Now process the array
    if isinstance(map_array, np.ndarray):
        map_array = np.where((map_array == 2) | (map_array == 3), 1, map_array)
        map_array = map_array.tolist()
    elif isinstance(map_array, list):
        map_array = [[1 if (tile == 2 or tile == 3) else tile for tile in row] for row in map_array]
    else:
        map_array = np.array(map_array)
        map_array = np.where((map_array == 2) | (map_array == 3), 1, map_array)
        map_array = map_array.tolist()

    game_dict = {
        'tiles':map_array,
        'player_spawn':[player_x, player_y],
        'stairs_spawn':[stairs_x, stairs_y],
        'width':56,
        'height':32,
        'difficulty':difficulty,
        'enemies':enemies
    }

    json_string = json.dumps(game_dict)

    return prompt, json_string

In [ ]:
# Test the function
example = dataset['train'][0]  # Get first training example
prompt, json_output = format_map_for_training(example)

print("Generated Prompt:")
print(prompt)
print("\n" + "="*50 + "\n")
print("JSON Output (first 500 chars):")
print(json_output[:500])
print("...")

# Verify JSON is valid
try:
    parsed = json.loads(json_output)
    print("\n✓ JSON is valid!")
    print("Keys:", list(parsed.keys()))
    print("Tiles array shape:", len(parsed['tiles']), "x", len(parsed['tiles'][0]))
    print("Player spawn:", parsed['player_spawn'])
    print("Stairs spawn:", parsed['stairs_spawn'])
    print("Enemies count:", len(parsed['enemies']))
except json.JSONDecodeError as e:
    print("\n✗ JSON is invalid:", e)

In [ ]:
class MapDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_context_length = 6144):
        self.hf_dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_context_length = max_context_length

    def __len__(self):
        return len(self.hf_dataset)
    
    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        prompt, json_output = format_map_for_training(example)

        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": json_output}
        ] 
        
        user_only = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=False
        )
        user_length = user_only['input_ids'].shape[1]

        full_tokenized = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            max_length=self.max_context_length,
            truncation=True,
            padding="max_length"
        )
        input_ids = full_tokenized['input_ids'].squeeze(0)
        attention_mask = full_tokenized['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[:user_length] = -100

        pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        padding_mask = (input_ids == pad_token_id)
        labels[padding_mask] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

In [ ]:
# Test MapDataset
print("=" * 60)
print("Testing MapDataset")
print("=" * 60)

# Create dataset instance
train_dataset_wrapped = MapDataset(
    hf_dataset=dataset['train'],
    tokenizer=tokenizer,
    max_context_length=6144
)

print(f"\n✓ Dataset created successfully")
print(f"  Dataset length: {len(train_dataset_wrapped)}")
print(f"  Max context length: {train_dataset_wrapped.max_context_length}")

# Test getting a single item
print(f"\n{'=' * 60}")
print("Testing __getitem__")
print("=" * 60)

try:
    sample = train_dataset_wrapped[0]
    
    print(f"\n✓ Successfully retrieved sample")
    print(f"  Keys: {list(sample.keys())}")
    print(f"  input_ids shape: {sample['input_ids'].shape}")
    print(f"  attention_mask shape: {sample['attention_mask'].shape}")
    print(f"  labels shape: {sample['labels'].shape}")
    
    # Check types
    print(f"\n  Types:")
    print(f"    input_ids type: {type(sample['input_ids'])}")
    print(f"    attention_mask type: {type(sample['attention_mask'])}")
    print(f"    labels type: {type(sample['labels'])}")
    
    # Verify all sequences are the same length (padded to max_length)
    expected_length = train_dataset_wrapped.max_context_length
    actual_length = sample['input_ids'].shape[0]
    print(f"\n  Length check:")
    print(f"    Expected length (max_context_length): {expected_length}")
    print(f"    Actual length: {actual_length}")
    if actual_length == expected_length:
        print(f"    ✓ All sequences padded to max_length")
    else:
        print(f"    ⚠ Warning: Length mismatch!")
    
    # Check label masking
    print(f"\n  Label masking check:")
    labels = sample['labels']
    num_masked = (labels == -100).sum().item()
    num_total = labels.numel()
    num_unmasked = num_total - num_masked
    print(f"    Total tokens: {num_total}")
    print(f"    Masked tokens: {num_masked}")
    print(f"    Unmasked tokens (assistant response): {num_unmasked}")
    print(f"    Masking ratio: {num_masked/num_total*100:.1f}%")
    
    # Verify masking is at the start (prompt) and padding
    if num_masked > 0:
        # Find first unmasked token
        unmasked_indices = (labels != -100).nonzero(as_tuple=True)[0]
        if len(unmasked_indices) > 0:
            first_unmasked = unmasked_indices[0].item()
            last_unmasked = unmasked_indices[-1].item()
            print(f"    First unmasked token at index: {first_unmasked}")
            print(f"    Last unmasked token at index: {last_unmasked}")
            print(f"    ✓ Prompt tokens masked (indices 0-{first_unmasked-1})")
            
            # Check if there's padding at the end
            if last_unmasked < actual_length - 1:
                padding_start = last_unmasked + 1
                print(f"    ✓ Padding tokens masked (indices {padding_start}-{actual_length-1})")
    
    # Decode a sample to verify format (skip padding tokens)
    print(f"\n  Decoded sample (first 200 non-padding tokens):")
    # Find where actual content ends (before padding)
    attention_mask = sample['attention_mask']
    content_end = attention_mask.sum().item()  # Sum gives number of non-padding tokens
    tokens_to_decode = min(200, content_end)
    decoded = tokenizer.decode(sample['input_ids'][:tokens_to_decode], skip_special_tokens=False)
    print(f"    {decoded[:500]}...")
    
    # Check that labels match input_ids where not masked
    print(f"\n  Label consistency check:")
    unmasked_mask = (labels != -100)
    if unmasked_mask.any():
        unmasked_labels = labels[unmasked_mask]
        unmasked_inputs = sample['input_ids'][unmasked_mask]
        matches = (unmasked_labels == unmasked_inputs).all().item()
        print(f"    Unmasked labels match input_ids: {matches}")
        if not matches:
            print(f"    ⚠ Warning: Some unmasked labels don't match input_ids!")
            # Show first few mismatches
            mismatches = (unmasked_labels != unmasked_inputs).nonzero(as_tuple=True)[0]
            if len(mismatches) > 0:
                print(f"    First mismatch at unmasked index: {mismatches[0].item()}")
    else:
        print(f"    ⚠ Warning: All labels are masked!")
    
    # Check padding token masking
    print(f"\n  Padding token check:")
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    padding_tokens = (sample['input_ids'] == pad_token_id).sum().item()
    padding_labels_masked = ((sample['input_ids'] == pad_token_id) & (labels == -100)).sum().item()
    print(f"    Padding tokens in input_ids: {padding_tokens}")
    print(f"    Padding tokens masked in labels: {padding_labels_masked}")
    if padding_tokens == padding_labels_masked:
        print(f"    ✓ All padding tokens are masked in labels")
    else:
        print(f"    ⚠ Warning: Some padding tokens not masked in labels")
    
    print(f"\n✓ All basic checks passed!")
    
except Exception as e:
    print(f"\n✗ Error retrieving sample: {e}")
    import traceback
    traceback.print_exc()

# Test multiple samples to verify consistent length
print(f"\n{'=' * 60}")
print("Testing Multiple Samples (Length Consistency)")
print("=" * 60)

try:
    # Get a few samples
    num_test_samples = min(5, len(train_dataset_wrapped))
    samples = [train_dataset_wrapped[i] for i in range(num_test_samples)]
    
    lengths = [s['input_ids'].shape[0] for s in samples]
    all_same_length = all(l == lengths[0] for l in lengths)
    
    print(f"\n  Tested {num_test_samples} samples")
    print(f"  Lengths: {lengths}")
    if all_same_length:
        print(f"  ✓ All samples have the same length: {lengths[0]}")
    else:
        print(f"  ⚠ Warning: Samples have different lengths!")
    
except Exception as e:
    print(f"\n✗ Error testing multiple samples: {e}")
    import traceback
    traceback.print_exc()

# Test data collator (optional, since we're padding in tokenizer)
print(f"\n{'=' * 60}")
print("Testing with DataCollator (Optional)")
print("=" * 60)

try:
    # Get a small batch
    batch_samples = [train_dataset_wrapped[i] for i in range(min(2, len(train_dataset_wrapped)))]
    
    print(f"  Sample shapes before collation:")
    for i, s in enumerate(batch_samples):
        print(f"    Sample {i}: input_ids={s['input_ids'].shape}, labels={s['labels'].shape}")
    
    # Apply data collator (should work since all sequences are same length)
    collated = data_collator(batch_samples)
    
    print(f"\n✓ Data collator works!")
    print(f"  Batch input_ids shape: {collated['input_ids'].shape}")
    print(f"  Batch attention_mask shape: {collated['attention_mask'].shape}")
    print(f"  Batch labels shape: {collated['labels'].shape}")
    print(f"  Expected batch size: {len(batch_samples)}")
    print(f"  Actual batch size: {collated['input_ids'].shape[0]}")
    
except Exception as e:
    print(f"\n✗ Error with data collator: {e}")
    print(f"  Note: This is okay if you're not using a data collator")
    import traceback
    traceback.print_exc()

print(f"\n{'=' * 60}")
print("Dataset testing complete!")
print("=" * 60)

In [ ]:
train_dataset = MapDataset(
    hf_dataset=dataset['train'],
    tokenizer=tokenizer,
    max_context_length=6144
)

val_dataset = MapDataset(
    hf_dataset=dataset['validation'],
    tokenizer=tokenizer,
    max_context_length=6144
)

In [ ]:
def train(
    model,
    tokenizer,
    batch_size,
    num_epochs,
    learning_rate,
    device,
    train_dataset,
    val_dataset,
    lora_config,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    eval_steps=None,
    save_steps=None,
    output_dir=None
):
    """ Main Training Loop

    Args:
        model (AutoModelForCausalLM): Model to train
        tokenizer (AutoTokenizer): Tokenizer for the model
        batch_size (int): Batch size
        num_epochs (int): Number of epochs to train
        learning_rate (float): Learning rate
        device (torch.device): Device to train on
        train_dataset: Training dataset
        val_dataset: Validation dataset
        lora_config: LoRA configuration
        gradient_accumulation_steps (int): Number of steps to accumulate gradients
        max_grad_norm (float): Maximum gradient norm for clipping
        eval_steps (int): Evaluate every N steps (None = end of epoch)
        save_steps (int): Save checkpoint every N steps (None = end of epoch)
        output_dir (str): Directory to save checkpoints
    """
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    model = get_peft_model(model, lora_config)
    model = model.to(device)
    model.print_trainable_parameters()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    num_training_steps = len(train_dataloader) * num_epochs
    
    global_step = 0
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model.forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            loss = loss / gradient_accumulation_steps
            loss.backward()

            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=max_grad_norm
                )

                optimizer.step()
                optimizer.zero_grad()

                global_step += 1

                if eval_steps and global_step % eval_steps == 0:
                    val_loss = evaluate(model, val_dataloader, device)
                    model.train()
                
                if save_steps and global_step % save_steps == 0:
                    if output_dir:
                        checkpoint_path = f"{output_dir}/checkpoint-{global_step}"
                        model.save_pretrained(checkpoint_path)
                        print(f"Saved checkpoint to {checkpoint_path}")
        
            epoch_loss += loss.item() * gradient_accumulation_steps
            num_batches += 1
    
        avg_epoch_loss = epoch_loss / num_batches
        print(f"\nEpoch {epoch+1} completed. Average loss: {avg_epoch_loss:.4f}")

        val_loss = evaluate(model, val_dataloader, device)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            if output_dir:
                best_model_path = f"{output_dir}/best_model"
                model.save_pretrained(best_model_path)
                print(f"Saved best model (val_loss={val_loss:.4f}) to {best_model_path}")
    
    return model

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            num_batches += 1
    avg_loss = total_loss / num_batches
    return avg_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

trained_model = train(
    model=model,
    tokenizer=tokenizer,
    batch_size=4,
    num_epochs=1,
    learning_rate=2e-4,
    device=device,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    lora_config=lora_config,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    eval_steps=50,
    save_steps=100,
    output_dir="./mysterydungeonGPT/trained_model/qwen3-lora-finetuned"
)